In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('house-prices-advanced-regression-techniques')

print("Path to competition files:", path)

## Import the library

In [ ]:
!pip install ydf -U
import ydf
import tensorflow as tf
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

%matplotlib inline

print("TensorFlow v" + tf.__version__)

In [ ]:
dataset_df = pd.read_csv(path+"/train.csv")
print("Full train dataset shape is {}".format(dataset_df.shape))
dataset_df_test = pd.read_csv(path+"/test.csv")
print("Full test dataset shape is {}".format(dataset_df_test.shape))

In [ ]:
dataset_df.head()


In [ ]:
dataset_df_test.head()

* There are 79 features columns. Using these features your model has to predict the house sale price indicated by the label column name SalePice.

We will drop the Id column as it is not necessary for model training.

In [ ]:
dataset_df = dataset_df.drop('Id', axis=1)
dataset_df.head(3)

We can inspect the types of feature columns using the following code:

In [ ]:
dataset_df.info()

## House Price Distribution
Now let us take a look at how the house prices are distributed.

In [ ]:
print(dataset_df['SalePrice'].describe())
plt.figure(figsize=(9,8))
sns.distplot(dataset_df['SalePrice'], color='g', bins=100, hist_kws = {'alpha': 0.4});

In [ ]:
list(set(dataset_df.dtypes.tolist()))

In [ ]:
df_num = dataset_df.select_dtypes(include = ['float64','int64'])
df_num.head()

In [ ]:
df_num.hist(figsize=(16, 20), bins=50, xlabelsize=8, ylabelsize=8);

## Prepare the dataset

This dataset coontains a mix of numeric, categorical and missing features. TF-DF supports all these feature types natively, and no preprocessing is required. This is one advantage of tree-based models, making them a great entry point to Tensorlow and ML.

Now let us split the dataset into training and testing datasets:

In [ ]:
import numpy as np

def split_dataset(dataset, test_ratio=0.3):
    test_indices = np.random.rand(len(dataset)) < test_ratio
    return dataset[~test_indices], dataset[test_indices]

train_ds_pd, valid_ds_pd = split_dataset(dataset_df)
print("{} examples in training, {} examples in testing.".format(
    len(train_ds_pd), len(valid_ds_pd)))

There's one more step required before we can train the model. We need to convert the datatset from Pandas format (pd.DataFrame) into TensorFlow Datasets format (tf.data.Dataset).

TensorFlow Datasets is a high performance data loading library which is helpful when training neural networks with accelerators like GPUs and TPUs.

linkcode
By default the Random Forest Model is configured to train classification tasks. Since this is a regression problem, we will specify the type of the task (tfdf.keras.Task.REGRESSION) as a parameter here.

In [ ]:
label = 'SalePrice'

## Select a Model

There are several tree-based models for you to choose from.
* RandomForestModel
* radientBoostedTreesModel
* CartModel
* DistributedGradientBoostedTreesModel

To start, we'll work with a Random Forest. This is the most well-known of the Decision Forest training algorithms.

A Random Forest is a collection of decision trees, each trained independently on a random subset of the training dataset (sampled with replacement). The algorithm is unique in that it is robust to overfitting, and easy to use.

Create a Random Forest
Today, we will use the defaults to create the Random Forest Model.

In [ ]:
model= ydf.RandomForestLearner(label=label).train(dataset_df)

In [ ]:
# Look at a model (input features, training logs, structure, etc.)
model.describe()

In [ ]:
# Generate predictions
model.predict(dataset_df)

In [ ]:
## Evaluate a model (e.g. roc, accuracy, confusion matrix, confidence intervals)
evaluation = model.evaluate(dataset_df)
# Query individual evaluation metrics
print(f"Test accuracy: {evaluation.accuracy}")

# Show the full evaluation report
print("Full evaluation report:")
evaluation

In [ ]:
# Analyse a model (e.g. partial dependence plot, variable importance)
model.analyze(dataset_df)

In [ ]:
# Benchmark the inference speed of a model
model.benchmark(dataset_df)